In [ ]:
# Step 1: Mount Google Drive (for saving checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies and clone repo
!pip install -q gsplat torchmetrics lpips tensorboard pyyaml

import os
if not os.path.exists('/content/GaussianFractalLOD'):
    !git clone https://github.com/dedoubleyou1/GaussianFractalLOD.git

%cd /content/GaussianFractalLOD

# Pull LFS data (NeRF Synthetic scenes)
!apt-get install -y git-lfs -qq
!git lfs install
!git lfs pull

!pip install -e . -q
print('Setup complete!')

In [ ]:
# NeRF Synthetic data is included in the repo (via Git LFS)
# All 8 scenes: chair, drums, ficus, hotdog, lego, materials, mic, ship
import os
REPO_DIR = os.getcwd()  # Should be /content/GaussianFractalLOD
DATA_DIR = os.path.join(REPO_DIR, 'nerf_synthetic')
assert os.path.exists(DATA_DIR), f'Data not found at {DATA_DIR}'
!ls {DATA_DIR}/

In [ ]:
from gaussianfractallod.config import Config

SCENE = "lego"  # Change for different scenes

cfg = Config(
    data_dir=f"{DATA_DIR}/{SCENE}",
    num_roots=32,
    sh_degree=0,  # Start with SH0 for fast iteration
    max_binary_depth=6,  # Start shallow, increase later
    root_iterations=5000,
    split_iterations_per_level=3000,
    checkpoint_dir=f"/content/drive/MyDrive/GaussianFractalLOD/checkpoints/{SCENE}",
)
print(f"Training {SCENE} with {cfg.num_roots} roots, depth {cfg.max_binary_depth}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from gaussianfractallod.train import train
from pathlib import Path

# To resume from a checkpoint, set this path:
RESUME_FROM = None  # e.g., f"{cfg.checkpoint_dir}/phase2_level_3.pt"

roots, tree = train(cfg, resume_from=RESUME_FROM)
print(f"Training complete! {tree.num_splits} splits across {tree.depth} levels")

In [ ]:
import torch
from gaussianfractallod.data import NerfSyntheticDataset
from gaussianfractallod.eval import evaluate

device = torch.device("cuda")
test_dataset = NerfSyntheticDataset(cfg.data_dir, split="test")
background = torch.tensor(cfg.background_color, device=device)

results = {}
for depth in range(tree.depth + 1):
    r = evaluate(roots.to(device), tree.to(device), test_dataset, depth, device, background)
    results[depth] = r
    print(f"Depth {depth}: PSNR={r['psnr']:.2f}, {r['num_gaussians']} Gaussians")

In [ ]:
import matplotlib.pyplot as plt
from gaussianfractallod.reconstruct import reconstruct
from gaussianfractallod.render import render_gaussians

# Pick a test view
gt_image, camera = test_dataset[0]
camera = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in camera.items()}

fig, axes = plt.subplots(1, min(tree.depth + 1, 7), figsize=(24, 4))
depths = list(range(min(tree.depth + 1, 7)))

for ax, depth in zip(axes, depths):
    with torch.no_grad():
        gaussians = reconstruct(roots.to(device), tree.to(device), depth)
        rendered = render_gaussians(
            gaussians, camera["viewmat"], camera["K"],
            camera["width"], camera["height"], background,
        )
    ax.imshow(rendered.cpu().numpy().clip(0, 1))
    ax.set_title(f"Depth {depth}\n{gaussians.num_gaussians} G")
    ax.axis("off")

plt.suptitle(f"LoD Progression \u2014 {SCENE}")
plt.tight_layout()
plt.savefig(f"{cfg.checkpoint_dir}/lod_progression.png", dpi=150)
plt.show()